In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("irfs_data")
SUFFIX = "_form_inventory"

# All four sheets per company: all_sheets["citizens"]["Triage"], etc.
all_sheets = {}
for path in sorted(DATA_DIR.glob("*.xlsx")):
    if path.name.startswith("~$"):  # skip Excel lock files
        continue
    company = path.stem.removesuffix(SUFFIX)
    all_sheets[company] = pd.read_excel(path, sheet_name=None)

# One variable per company holding its Inventory sheet: american_integrity, citizens, ...
for company, sheets in all_sheets.items():
    globals()[company] = sheets["Inventory"]


In [7]:
# Clean up datetime columns in Inventory sheets
DATE_COLUMNS = ["Doc Date", "Filed", "Eff Date (New)"]  # stored as "MM/DD/YYYY" text

for sheets in all_sheets.values():
    inv = sheets["Inventory"]  # same object as the per-company variables, so they update too
    for col in DATE_COLUMNS:
        inv[col] = pd.to_datetime(inv[col], format="%m/%d/%Y")

In [8]:
summary = pd.DataFrame(
    {company: {name: len(df) for name, df in sheets.items()} for company, sheets in all_sheets.items()}
).T

summary["filed_min"] = pd.Series({company: sheets["Inventory"]["Filed"].min() for company, sheets in all_sheets.items()})
summary["filed_max"] = pd.Series({company: sheets["Inventory"]["Filed"].max() for company, sheets in all_sheets.items()})
summary["timeframe"] = summary["filed_min"].dt.strftime("%Y-%m-%d") + " to " + summary["filed_max"].dt.strftime("%Y-%m-%d")

summary

,Inventory,Triage,Trade Secrets,Summary,filed_min,filed_max,timeframe
american_integrity,3425,470,18,12,2006-09-20,2026-03-20,2006-09-20 to 2026-03-20
citizens,7038,899,0,12,2003-02-17,2026-06-18,2003-02-17 to 2026-06-18
first_protective,1527,287,1,11,2003-02-25,2025-08-05,2003-02-25 to 2025-08-05
florida_peninsula,5062,831,418,12,2005-06-02,2026-06-09,2005-06-02 to 2026-06-09
heritage,2506,486,0,12,2012-07-17,2026-01-13,2012-07-17 to 2026-01-13
homeowners_choice,1742,176,11,12,2007-03-22,2025-08-08,2007-03-22 to 2025-08-08
manatee,385,288,66,12,2024-02-02,2025-05-27,2024-02-02 to 2025-05-27
slide_insurance,513,349,166,12,2022-02-23,2026-06-15,2022-02-23 to 2026-06-15
state_farm,1520,103,0,12,2001-11-15,2025-04-28,2001-11-15 to 2025-04-28
tower_hill,3621,624,49,12,2002-04-23,2026-06-26,2002-04-23 to 2026-06-26


In [15]:
# Q1: unique filings and share of approved, by year filed
inv = pd.concat([sheets["Inventory"] for sheets in all_sheets.values()])  # all 12 insurers in one table
filings = inv.drop_duplicates("File Log #")  # one row per filing
filings_per_year = filings["Filed"].dt.year.value_counts().sort_index()

year = filings["Filed"].dt.year
approved_per_year = year[filings["Final Action"] == "APPROVED"].value_counts().sort_index()
withdrawn_per_year = year[filings["Final Action"] == "WITHDRAWN"].value_counts().sort_index()

# filed, approved and withdrawn side by side
q1 = pd.DataFrame({"filed": filings_per_year, "approved": approved_per_year, "withdrawn": withdrawn_per_year}).fillna(0).astype(int)
q1["approved_ratio"] = (q1["approved"] / q1["filed"]).round(3)
q1


,filed,approved,withdrawn,approved_ratio
Filed,,,,
2001,2,0,0,0.000
2002,11,9,0,0.818
2003,24,19,0,0.792
2004,13,11,2,0.846
2005,45,40,2,0.889
2006,33,22,4,0.667
2007,67,58,7,0.866
2008,84,71,8,0.845
2009,73,63,6,0.863


In [14]:
# Q2: unique forms and share of approved, by year
FORM_DOC_TYPES = ["Forms", "Policy Forms", "FORM", "Revised form", "Revised form markup", "Forms to be Reviewed"]
forms = inv[inv["Doc Type"].isin(FORM_DOC_TYPES)].copy()

# Form name: the Form #, or the title when Form # is "*" (trade secret) or blank; ignore "marked"/"clean" and spaces
forms["form"] = forms["Form #"].replace("*", None).fillna(forms["Document Title"])
forms["form"] = forms["form"].str.replace(r"(?i)marked|clean|\s+", "", regex=True).str.upper()

# Drop marked/clean copies that have no status of their own (redlines, not forms OIR acted on)
working_copy = forms["Document Title"].str.contains("marked|clean", case=False, na=False)
forms = forms[~(working_copy & forms["Doc Status"].isna())]

# One row per form in each filing: its latest version; blank status -> the filing's outcome
forms = forms.sort_values("Doc Date").drop_duplicates(["File Log #", "form"], keep="last")
status = forms["Doc Status"].fillna(forms["Final Action"])

year = forms["Filed"].dt.year
q2 = pd.DataFrame({
    "forms": year.value_counts(),
    "approved": year[status == "APPROVED"].value_counts(),
    "withdrawn": year[status == "WITHDRAWN"].value_counts(),
}).fillna(0).astype(int).sort_index()
q2["approved_ratio"] = (q2["approved"] / q2["forms"]).round(3)
q2


,forms,approved,withdrawn,approved_ratio
Filed,,,,
2001,6,5,0,0.833
2002,24,20,1,0.833
2003,58,45,0,0.776
2004,94,85,9,0.904
2005,332,264,56,0.795
2006,241,186,48,0.772
2007,578,500,31,0.865
2008,418,374,37,0.895
2009,198,172,18,0.869


In [18]:
# Q1 + Q2 in one table, years as columns, plus totals for each period
PERIODS = {"2016–2020": (2016, 2020), "2021–2025": (2021, 2025)}  # five years vs five years

counts = pd.concat([
    q1[["filed", "approved", "withdrawn"]].rename(columns={"filed": "Filings filed", "approved": "Filings approved",
                                                           "withdrawn": "Filings withdrawn"}),
    q2[["forms", "approved", "withdrawn"]].rename(columns={"forms": "Policy forms", "approved": "Policy forms approved",
                                                           "withdrawn": "Policy forms withdrawn"}),
], axis=1)

first_year = min(start for start, end in PERIODS.values())
last_year = max(end for start, end in PERIODS.values())
combined = counts.loc[first_year:last_year].copy()
for label, (start, end) in PERIODS.items():
    combined.loc[label] = counts.loc[start:end].sum()  # period total

# Ratios from the counts, so period ratios are total approved / total filed
combined.insert(3, "Filings approved %", (combined["Filings approved"] / combined["Filings filed"]).map("{:.1%}".format))
combined["Policy forms approved %"] = (combined["Policy forms approved"] / combined["Policy forms"]).map("{:.1%}".format)

combined.T

Filed,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2016–2020,2021–2025
Filings filed,73,54,63,43,29,50,63,96,79,52,262,340
Filings approved,52,43,49,31,24,49,55,94,77,48,199,323
Filings withdrawn,19,9,10,11,5,1,7,1,1,2,54,12
Filings approved %,71.2%,79.6%,77.8%,72.1%,82.8%,98.0%,87.3%,97.9%,97.5%,92.3%,76.0%,95.0%
Policy forms,430,311,470,184,185,401,387,330,453,176,1580,1747
Policy forms approved,310,236,421,137,139,395,232,317,401,125,1243,1470
Policy forms withdrawn,118,72,48,45,46,6,144,8,21,22,329,201
Policy forms approved %,72.1%,75.9%,89.6%,74.5%,75.1%,98.5%,59.9%,96.1%,88.5%,71.0%,78.7%,84.1%


In [19]:
# Same table without Slide's and Manatee's first-year filings (new carriers filing their whole program)
FIRST_YEAR = {"SLIDE INSURANCE COMPANY": 2022, "MANATEE INSURANCE EXCHANGE": 2024}
is_first_year = filings["Entity"].map(FIRST_YEAR) == filings["Filed"].dt.year
excluded = filings.loc[is_first_year, "File Log #"]

kept_filings = filings[~filings["File Log #"].isin(excluded)]
kept_forms = forms[~forms["File Log #"].isin(excluded)]
filing_year, form_year = kept_filings["Filed"].dt.year, kept_forms["Filed"].dt.year
form_status = kept_forms["Doc Status"].fillna(kept_forms["Final Action"])  # same status rule as Q2

counts_excl = pd.DataFrame({
    "Filings filed": filing_year.value_counts(),
    "Filings approved": filing_year[kept_filings["Final Action"] == "APPROVED"].value_counts(),
    "Filings withdrawn": filing_year[kept_filings["Final Action"] == "WITHDRAWN"].value_counts(),
    "Policy forms": form_year.value_counts(),
    "Policy forms approved": form_year[form_status == "APPROVED"].value_counts(),
    "Policy forms withdrawn": form_year[form_status == "WITHDRAWN"].value_counts(),
}).fillna(0).astype(int).sort_index()

combined_excl = counts_excl.loc[first_year:last_year].copy()
for label, (start, end) in PERIODS.items():
    combined_excl.loc[label] = counts_excl.loc[start:end].sum()  # period total

combined_excl.insert(3, "Filings approved %", (combined_excl["Filings approved"] / combined_excl["Filings filed"]).map("{:.1%}".format))
combined_excl["Policy forms approved %"] = (combined_excl["Policy forms approved"] / combined_excl["Policy forms"]).map("{:.1%}".format)

print("Excluded filings:", len(excluded))
combined_excl.T

Excluded filings: 18


Filed,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2016–2020,2021–2025
Filings filed,73,54,63,43,29,50,55,96,69,52,262,322
Filings approved,52,43,49,31,24,49,49,94,67,48,199,307
Filings withdrawn,19,9,10,11,5,1,5,1,1,2,54,10
Filings approved %,71.2%,79.6%,77.8%,72.1%,82.8%,98.0%,89.1%,97.9%,97.1%,92.3%,76.0%,95.3%
Policy forms,430,311,470,184,185,401,198,330,232,176,1580,1337
Policy forms approved,310,236,421,137,139,395,164,317,201,125,1243,1202
Policy forms withdrawn,118,72,48,45,46,6,23,8,10,22,329,69
Policy forms approved %,72.1%,75.9%,89.6%,74.5%,75.1%,98.5%,82.8%,96.1%,86.6%,71.0%,78.7%,89.9%


In [22]:
# number of filings stamped trade-secret, by year
# A filing counts if any of its documents has Trade Secret = "YES"
ts_filings = inv.loc[inv["Trade Secret"] == "YES", "File Log #"].unique()
ts_year = filings.loc[filings["File Log #"].isin(ts_filings), "Filed"].dt.year

trade_secret = pd.DataFrame({
    "Filings filed": filings_per_year,
    "Trade-secret filings": ts_year.value_counts(),
}).fillna(0).astype(int)
trade_secret["Trade-secret %"] = (trade_secret["Trade-secret filings"] / trade_secret["Filings filed"]).map("{:.1%}".format)

trade_secret.loc[2016:2026].T

Filed,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Filings filed,73,54,63,43,29,50,63,96,79,52,19
Trade-secret filings,0,0,0,0,0,2,2,9,20,20,10
Trade-secret %,0.0%,0.0%,0.0%,0.0%,0.0%,4.0%,3.2%,9.4%,25.3%,38.5%,52.6%
